# Stage 1 risk-score: Decision Tree Regressor

This is the nonlinear challenger to the Linear Regression risk-score stage. It uses the same
collision-level data and grouped-safe out-of-fold process, so the two regressors can be compared
fairly before the final ensemble notebook chooses its risk input. Run all cells on Windows after
pushing; this review does not execute the full grid.

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV, KFold, cross_val_predict
import sys
sys.path.insert(0, str(Path.cwd() / 'hyper parameter')); sys.path.insert(0, str(Path.cwd()))
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if not (ROOT / 'dataset').exists(): ROOT = ROOT.parent
TRAIN_PATH = ROOT / 'dataset' / 'processed' / 'regression_train.csv'
TEST_PATH = ROOT / 'dataset' / 'processed' / 'regression_test.csv'
RESULTS = ROOT / 'hyper parameter' / 'results' / 'decision_tree_regressor'
BEST = ROOT / 'hyper parameter' / 'best'
MODEL_DIR = ROOT / 'models' / 'saved_models'
RESULTS.mkdir(parents=True, exist_ok=True); BEST.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)
SEED = 19
TARGET = 'number_of_casualties'
GROUP = 'collision_index'
FEATURES = ['number_of_vehicles', 'day_of_week', 'first_road_class', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area', 'special_conditions_at_site', 'carriageway_hazards']
NUMERIC = ['number_of_vehicles', 'speed_limit']
CATEGORICAL = [c for c in FEATURES if c not in NUMERIC]

In [2]:
train = pd.read_csv(TRAIN_PATH, dtype={GROUP: str})
test = pd.read_csv(TEST_PATH, dtype={GROUP: str})
X, y = train[FEATURES], train[TARGET].astype(float)
X_test, y_test = test[FEATURES], test[TARGET].astype(float)
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
    ('numeric', 'passthrough', NUMERIC),
], remainder='drop')
pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', DecisionTreeRegressor(random_state=SEED))])
PARAM_GRID = {
    'regressor__max_depth': [None, 3, 5, 8, 12],
    'regressor__min_samples_leaf': [1, 2, 5, 10],
    'regressor__min_samples_split': [2, 10, 25],
    'regressor__max_features': [None, 'sqrt', 0.7],
    'regressor__criterion': ['squared_error', 'absolute_error'],
    'regressor__ccp_alpha': [0.0, 0.0001, 0.001],
}
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print('collision-level train:', X.shape, 'test:', X_test.shape)
print('parameter combinations:', 5 * 4 * 3 * 3 * 2 * 3)

collision-level train: (81058, 13) test: (20467, 13)
parameter combinations: 1080


In [3]:
scoring = {'r2': 'r2', 'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error'}
search = RandomizedSearchCV(pipeline, PARAM_GRID, scoring=scoring, refit='r2', n_iter=16, random_state=SEED, cv=cv, n_jobs=1, verbose=2, return_train_score=True, error_score='raise')
search.fit(X, y)
cv_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_r2')
cv_results.to_csv(RESULTS / 'cv_results.csv', index=False)
print('best parameters:', search.best_params_)
print('best CV R2:', search.best_score_)

Fitting 5 folds for each of 1080 candidates, totalling 5400 fits


KeyboardInterrupt: 

In [ ]:
best_model = search.best_estimator_
oof = cross_val_predict(best_model, X, y, cv=cv, method='predict', n_jobs=1)
test_prediction = best_model.predict(X_test)
pd.DataFrame({GROUP: train[GROUP], 'risk_score': oof}).to_csv(RESULTS / 'oof_predictions.csv', index=False)
pd.DataFrame({GROUP: test[GROUP], 'risk_score': test_prediction}).to_csv(RESULTS / 'test_predictions.csv', index=False)
metrics = {
    'model': 'decision_tree_regressor', 'features': FEATURES, 'best_params': search.best_params_,
    'cv_r2': float(search.best_score_), 'oof_rmse': float(np.sqrt(mean_squared_error(y, oof))),
    'oof_mae': float(mean_absolute_error(y, oof)), 'test_r2': float(r2_score(y_test, test_prediction)),
    'test_rmse': float(np.sqrt(mean_squared_error(y_test, test_prediction))),
    'test_mae': float(mean_absolute_error(y_test, test_prediction)),
}
(RESULTS / 'best_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
joblib.dump(best_model, BEST / 'decision_tree_regressor_best.pkl')
joblib.dump(best_model, MODEL_DIR / 'decision_tree_regressor_1.pkl')
(BEST / 'decision_tree_regressor_best_params.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))